# 02 - Dataset, fonti e copertura delle domande

Questo notebook esplora la parte metodologica del repository.

Domande operative:
- quali dataset logici servono;
- quali fonti li alimentano;
- quali indicatori sono richiesti;
- quali domande delle live sono gia' mappate;
- quali collegamenti tecnici mancano ancora.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
CODE_DIR = REPO_ROOT / 'code'
METADATA_DIR = REPO_ROOT / 'metadata'
DATA_FINAL_DIR = REPO_ROOT / 'data' / 'final'

if str(CODE_DIR) not in sys.path:
    sys.path.append(str(CODE_DIR))

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 120)

In [ ]:
def read_csv_optional(path: Path) -> pd.DataFrame:
    """Legge un CSV se esiste. Restituisce una tabella vuota se manca."""
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame()
    return pd.read_csv(path)


def split_semicolon(series: pd.Series) -> pd.Series:
    """Espande celle con valori separati da punto e virgola."""
    return (
        series.fillna('')
        .astype(str)
        .str.split(';')
        .explode()
        .str.strip()
        .loc[lambda x: x.ne('')]
    )

## Caricamento metadata

Queste tabelle sono il cuore della documentazione metodologica.

In [ ]:
fonti = read_csv_optional(METADATA_DIR / 'registro_fonti.csv')
dataset_attesi = read_csv_optional(METADATA_DIR / 'dataset_attesi.csv')
indicatori = read_csv_optional(METADATA_DIR / 'definizioni_indicatori.csv')
domande = read_csv_optional(METADATA_DIR / 'domande_live.csv')

print('Fonti:', len(fonti))
print('Dataset attesi:', len(dataset_attesi))
print('Indicatori:', len(indicatori))
print('Domande live:', len(domande))

## Dataset per fonte

Mostra quante unita' logiche sono previste per ogni fonte. Questo non indica ancora il numero di file scaricati. Indica il disegno analitico.

In [ ]:
if dataset_attesi.empty:
    print('dataset_attesi.csv non disponibile.')
else:
    per_fonte = (
        dataset_attesi
        .groupby(['fonte_id', 'ente'], dropna=False)
        .size()
        .reset_index(name='dataset_logici')
        .sort_values('dataset_logici', ascending=False)
    )
    display(per_fonte)

## Dataset per ambito

Questa vista aiuta a capire se il repository copre tutti i blocchi delle live.

In [ ]:
if dataset_attesi.empty:
    print('dataset_attesi.csv non disponibile.')
else:
    per_ambito = (
        dataset_attesi
        .groupby('ambito', dropna=False)
        .size()
        .reset_index(name='dataset_logici')
        .sort_values('dataset_logici', ascending=False)
    )
    display(per_ambito)

## Indicatori richiesti dai dataset

Qui si controlla se gli indicatori richiesti dai dataset sono presenti nel dizionario indicatori.

In [ ]:
if dataset_attesi.empty or indicatori.empty:
    print('Servono dataset_attesi.csv e definizioni_indicatori.csv.')
else:
    richiesti = split_semicolon(dataset_attesi['indicatori']).drop_duplicates().sort_values()
    definiti = set(indicatori['indicatore_id'].dropna().astype(str))
    check_indicatori = pd.DataFrame({'indicatore_id': richiesti})
    check_indicatori['definito'] = check_indicatori['indicatore_id'].isin(definiti)
    display(check_indicatori)

## Domande per tema

Questa vista mostra il peso dei blocchi tematici nella matrice live.

In [ ]:
if domande.empty:
    print('domande_live.csv non disponibile.')
else:
    domande_per_tema = (
        domande
        .groupby('tema', dropna=False)
        .size()
        .reset_index(name='domande')
        .sort_values('domande', ascending=False)
    )
    display(domande_per_tema)

## Copertura automatica

La copertura e' intenzionalmente severa. Una domanda non risulta coperta finche' la tabella finale non contiene l indicatore richiesto.

In [ ]:
try:
    from copertura_live import esegui_copertura_live

    copertura = esegui_copertura_live()
    display(copertura)

    if not copertura.empty:
        riepilogo = (
            copertura
            .groupby('stato', dropna=False)
            .size()
            .reset_index(name='domande')
            .sort_values('domande', ascending=False)
        )
        display(riepilogo)
except Exception as error:
    print('Copertura non eseguita.')
    print(type(error).__name__, error)

## Controllo coerenza del catalogo dataset

Questo controllo verifica fonti registrate, indicatori registrati, tabelle finali registrate e ID logici univoci.

In [ ]:
try:
    from dataset_attesi import controlla_dataset_attesi

    controllo_dataset = pd.DataFrame(controlla_dataset_attesi())
    display(controllo_dataset)
except Exception as error:
    print('Controllo dataset non eseguito.')
    print(type(error).__name__, error)

## Prossimo passaggio operativo

Per attivare il download automatico bisogna popolare le whitelist. La regola consigliata e' partire da `dataset_attesi.csv`, cercare il dataset tecnico nella fonte corrispondente, poi inserire nella whitelist l ID o l URL necessario.

Esempio logico:
- `inps_pensioni_vigenti` indica il bisogno analitico;
- `whitelist_inps.csv` deve contenere il dataset tecnico INPS effettivo;
- la trasformazione deve portare quel dato nelle tabelle finali previste.